# ECG Analysis — Heart Rate, HRV and Breathing Rate

Analyses BDF ECG files (L1 and L2 sensors) per patient.  For each sensor the
full recording is divided into non-overlapping 120-second windows.  Per-window
statistics include heart rate, ECG-derived respiration, and time/frequency-domain
HRV metrics (RMSSD, SDNN, LF power, HF power, LF/HF ratio).  Results are
displayed as raw time series and grouped by exercise stage.

**Pipeline**
1. Discover patient folders → find L1 / L2 BDF files.
2. Extract windowed features (`window_duration = 120 s`); cache to disk.
3. Annotate each window with its exercise stage from the telemetry CSV.
4. Per-patient: time-series table, time-series plot, stage pivot table, stage bar chart.
5. Cross-patient summary table.
6. Export CSVs.

In [ ]:
import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import functions as fn
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

fn.setup_logging(log_file="pipeline.log", level=logging.INFO)
logger = logging.getLogger("ecg_analysis")

In [ ]:
# ── Configure paths ──────────────────────────────────────────────────────────
DATA_ROOT: str = "data"
OUTPUT_ROOT: str = "output"
WINDOW_DURATION: float = 120.0  # seconds per analysis window

output_base = Path(OUTPUT_ROOT)
ecg_out = output_base / "ecg_analysis"
ecg_out.mkdir(parents=True, exist_ok=True)

logger.info("ECG analysis pipeline started.  Data: %s  Output: %s", DATA_ROOT, ecg_out)

## 1. Discover patient folders

In [ ]:
patient_folders = fn.discover_patient_folders(DATA_ROOT)
print(f"Found {len(patient_folders)} patient folder(s):")
for pf in patient_folders:
    print(" -", pf.name)

## 2. Extract ECG features

In [ ]:
# ── Stage-annotation helper ───────────────────────────────────────────────────

def _parse_time_to_seconds(t: str) -> float:
    """Parse a MM:SS or HH:MM:SS string to total seconds."""
    parts = str(t).strip().split(":")
    try:
        if len(parts) == 2:
            return float(parts[0]) * 60.0 + float(parts[1])
        if len(parts) == 3:
            return float(parts[0]) * 3600.0 + float(parts[1]) * 60.0 + float(parts[2])
    except ValueError:
        pass
    return float("nan")


def _annotate_stages(ecg_df: pd.DataFrame, telemetry_df: pd.DataFrame) -> pd.DataFrame:
    """Add a Stage column to *ecg_df* by nearest-match to telemetry time.

    Parameters
    ----------
    ecg_df : pd.DataFrame
        Output of :func:`fn.extract_ecg_timeseries`; must contain ``time_s``.
    telemetry_df : pd.DataFrame
        Cleaned telemetry DataFrame with a ``Stage`` column; first column is
        the HH:MM / MM:SS time string from the WESENSE export.

    Returns
    -------
    pd.DataFrame
        Copy of *ecg_df* with an added ``Stage`` column.
    """
    out = ecg_df.copy()

    if "Stage" not in telemetry_df.columns or telemetry_df.empty:
        out["Stage"] = pd.NA
        return out

    time_col = telemetry_df.columns[0]
    tele = telemetry_df[[time_col, "Stage"]].copy()
    tele["_t_s"] = tele[time_col].astype(str).apply(_parse_time_to_seconds)
    tele = tele.dropna(subset=["_t_s", "Stage"]).sort_values("_t_s")

    if tele.empty:
        out["Stage"] = pd.NA
        return out

    # Nearest-match each ECG window centre to the closest telemetry row.
    ecg_sorted = out.sort_values("time_s").copy()
    merged = pd.merge_asof(
        ecg_sorted,
        tele[["_t_s", "Stage"]].rename(columns={"_t_s": "time_s"}),
        on="time_s",
        direction="nearest",
    )
    return merged


# ── Extraction loop ───────────────────────────────────────────────────────────

STAGE_COLOURS = dict(zip(
    fn.STAGE_ORDER,
    ["#aec6cf", "#b5e7a0", "#ffb347", "#ff6961", "#c3b1e1"],
))

results: dict = {}  # patient_id -> {"L1": df, "L2": df, "telemetry": df}

n_total = len(patient_folders)
for i, folder in enumerate(tqdm(patient_folders, desc="Patients"), start=1):
    patient_id = folder.name
    print(f"\n[{i}/{n_total}] Patient: {patient_id}")
    logger.info("Processing ECG for patient: %s", patient_id)

    patient_out = ecg_out / patient_id
    patient_out.mkdir(parents=True, exist_ok=True)

    # Load telemetry (for stage annotation); gracefully handle missing CSV.
    telemetry_df: pd.DataFrame = pd.DataFrame()
    csv_path = fn.find_csv_file(folder)
    if csv_path is not None:
        try:
            _, telemetry_df = fn.load_telemetry(csv_path)
        except Exception as exc:
            logger.warning("  %s — could not load telemetry: %s", patient_id, exc)

    # Process L1 and L2 sensors.
    sensor_dfs: dict = {}
    l1_path, l2_path = fn.find_bdf_files(folder)
    for label, bdf_path in [("L1", l1_path), ("L2", l2_path)]:
        if bdf_path is None:
            logger.warning("  %s — %s BDF not found.", patient_id, label)
            continue

        # Try checkpoint first.
        ts_df = fn.load_ecg_checkpoint(patient_out, f"ts_{label}")
        if ts_df is None:
            raw = fn.load_ecg(bdf_path)
            if raw is None:
                logger.warning("  %s — failed to load %s BDF.", patient_id, label)
                continue
            ts_df = fn.extract_ecg_timeseries(raw, window_duration=WINDOW_DURATION)
            fn.save_ecg_checkpoint(ts_df, patient_out, f"ts_{label}")
            print(f"  {label}: {len(ts_df)} windows extracted")
        else:
            print(f"  {label}: loaded from checkpoint ({len(ts_df)} windows)")

        # Annotate with exercise stages.
        if not telemetry_df.empty:
            ts_df = _annotate_stages(ts_df, telemetry_df)

        sensor_dfs[label] = ts_df

    if sensor_dfs:
        results[patient_id] = {**sensor_dfs, "telemetry": telemetry_df}
    else:
        print(f"  No BDF data found for patient {patient_id}.")

print(f"\nExtraction complete.  Processed {len(results)} patient(s).")

## 3. Per-patient results

In [ ]:
# ── Plot helper: shade stage background regions ────────────────────────────
def _shade_stages(ax, df: pd.DataFrame, time_col: str = "time_s") -> None:
    """Shade background of *ax* according to the Stage column of *df*."""
    if "Stage" not in df.columns:
        return
    stage_col = df["Stage"].astype(str)
    times = df[time_col].values
    changes = np.where(np.diff(stage_col.values.astype(str), prepend="") != "")[0]
    for j, start_idx in enumerate(changes):
        end_idx = changes[j + 1] if j + 1 < len(changes) else len(times)
        stage_name = stage_col.iloc[start_idx]
        colour = STAGE_COLOURS.get(stage_name, "#eeeeee")
        ax.axvspan(times[start_idx], times[end_idx - 1], alpha=0.18,
                   color=colour, label=stage_name)


# ── Metric display names ──────────────────────────────────────────────────────
METRIC_LABELS: dict = {
    "hr_bpm": "Heart rate (bpm)",
    "breathing_rate_bpm": "Breathing rate (bpm)",
    "rmssd_ms": "RMSSD (ms)",
    "sdnn_ms": "SDNN (ms)",
    "lf_ms2": "LF power (ms²)",
    "hf_ms2": "HF power (ms²)",
    "lf_hf_ratio": "LF/HF ratio",
}

TABLE_COLS = ["time_s", "Stage", "hr_bpm", "breathing_rate_bpm",
              "rmssd_ms", "sdnn_ms", "lf_ms2", "hf_ms2", "lf_hf_ratio"]

HRV_METRICS = ["rmssd_ms", "sdnn_ms", "lf_ms2", "hf_ms2", "lf_hf_ratio"]

for patient_id, data in results.items():
    sensors = {k: v for k, v in data.items() if k in ("L1", "L2")}
    if not sensors:
        continue

    print("=" * 70)
    print(f"Patient: {patient_id}")
    print("=" * 70)

    patient_out = ecg_out / patient_id

    # ── 3a. Time-series tables ────────────────────────────────────────────
    print("\n### Time-series tables (one row = one 120-second window)")
    for label, df in sensors.items():
        available_cols = [c for c in TABLE_COLS if c in df.columns]
        display_df = df[available_cols].copy()
        display_df["time_s"] = display_df["time_s"].round(1)
        for col in HRV_METRICS + ["hr_bpm", "breathing_rate_bpm"]:
            if col in display_df.columns:
                display_df[col] = display_df[col].round(2)
        display_df = display_df.rename(columns=METRIC_LABELS)
        print(f"\n**Sensor {label}**")
        display(display_df)

    # ── 3b. Time-series plots (stage-shaded) ─────────────────────────────
    for label, df in sensors.items():
        if df.empty:
            continue

        # One channel per sensor (first channel only to keep the plot readable).
        channels = df["channel"].unique()
        for ch in channels:
            ch_df = df[df["channel"] == ch].sort_values("time_s")

            fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

            # Top: HR and breathing rate.
            ax0 = axes[0]
            ax0b = ax0.twinx()
            _shade_stages(ax0, ch_df)
            ax0.plot(ch_df["time_s"], ch_df["hr_bpm"], color="#2c7bb6",
                     linewidth=1.5, label="HR (bpm)")
            ax0b.plot(ch_df["time_s"], ch_df["breathing_rate_bpm"],
                      color="#d7191c", linewidth=1.5, linestyle="--",
                      label="Breathing rate (bpm)")
            ax0.set_ylabel("Heart rate (bpm)", color="#2c7bb6", fontsize=10)
            ax0b.set_ylabel("Breathing rate (bpm)", color="#d7191c", fontsize=10)
            # Combine legends.
            lines0, labels0 = ax0.get_legend_handles_labels()
            lines0b, labels0b = ax0b.get_legend_handles_labels()
            ax0.legend(lines0 + lines0b, labels0 + labels0b,
                       loc="upper right", fontsize=8)
            ax0.grid(True, alpha=0.3)
            ax0.set_title(f"Patient {patient_id} — Sensor {label} — Channel {ch}",
                          fontsize=11)

            # Bottom: HRV metrics.
            ax1 = axes[1]
            _shade_stages(ax1, ch_df)
            colours_hrv = ["#1a9641", "#756bb1", "#d73027"]
            hrv_plot = ["rmssd_ms", "sdnn_ms", "lf_hf_ratio"]
            hrv_colours = dict(zip(hrv_plot, colours_hrv))
            ax1b = ax1.twinx()
            for metric in ["rmssd_ms", "sdnn_ms"]:
                if metric in ch_df.columns:
                    ax1.plot(ch_df["time_s"], ch_df[metric],
                             color=hrv_colours[metric], linewidth=1.5,
                             label=METRIC_LABELS[metric])
            if "lf_hf_ratio" in ch_df.columns:
                ax1b.plot(ch_df["time_s"], ch_df["lf_hf_ratio"],
                          color=hrv_colours["lf_hf_ratio"], linewidth=1.5,
                          linestyle=":", label="LF/HF ratio")
            ax1.set_ylabel("RMSSD / SDNN (ms)", fontsize=10)
            ax1b.set_ylabel("LF/HF ratio", color=hrv_colours["lf_hf_ratio"],
                            fontsize=10)
            lines1, labels1 = ax1.get_legend_handles_labels()
            lines1b, labels1b = ax1b.get_legend_handles_labels()
            ax1.legend(lines1 + lines1b, labels1 + labels1b,
                       loc="upper right", fontsize=8)
            ax1.set_xlabel("Time (s)", fontsize=10)
            ax1.grid(True, alpha=0.3)

            # Stage legend patch (deduplicated).
            if "Stage" in ch_df.columns:
                seen = set()
                patches = []
                for stage in ch_df["Stage"].dropna().unique():
                    if stage not in seen:
                        patches.append(plt.Rectangle(
                            (0, 0), 1, 1,
                            fc=STAGE_COLOURS.get(str(stage), "#eeeeee"), alpha=0.5,
                            label=str(stage),
                        ))
                        seen.add(stage)
                if patches:
                    fig.legend(handles=patches, title="Stage",
                               loc="lower center", ncol=len(patches),
                               fontsize=8, bbox_to_anchor=(0.5, -0.02))

            fig.tight_layout(rect=(0, 0.04, 1, 1))
            save_path = patient_out / f"ecg_timeseries_{label}_{ch}.png"
            fig.savefig(save_path, dpi=150, bbox_inches="tight")
            plt.show()
            logger.info("Saved: %s", save_path)

    # ── 3c. Stage-aligned pivot table ────────────────────────────────────
    if any("Stage" in df.columns for df in sensors.values()):
        print("\n### Stage-aligned summary (mean per 120-second window)")
        pivot_parts = []
        for label, df in sensors.items():
            if "Stage" not in df.columns or df.empty:
                continue
            agg_cols = [c for c in HRV_METRICS + ["hr_bpm", "breathing_rate_bpm"]
                        if c in df.columns]
            pivot = (
                df.groupby("Stage", observed=True)[agg_cols]
                .mean()
                .round(2)
                .reindex([s for s in fn.STAGE_ORDER if s in df["Stage"].unique()])
            )
            pivot.columns = pd.MultiIndex.from_tuples(
                [(METRIC_LABELS.get(c, c), label) for c in pivot.columns]
            )
            pivot_parts.append(pivot)

        if pivot_parts:
            combined_pivot = pd.concat(pivot_parts, axis=1)
            display(combined_pivot)

    # ── 3d. Per-stage bar chart (RMSSD, SDNN, LF/HF) ─────────────────────
    bar_metrics = ["rmssd_ms", "sdnn_ms", "lf_hf_ratio"]
    bar_labels = ["RMSSD (ms)", "SDNN (ms)", "LF/HF ratio"]

    valid_sensors = {
        lbl: df for lbl, df in sensors.items()
        if "Stage" in df.columns and not df.empty
    }
    if valid_sensors:
        fig, axes = plt.subplots(1, len(bar_metrics),
                                 figsize=(5 * len(bar_metrics), 5))
        if len(bar_metrics) == 1:
            axes = [axes]

        sensor_colours = {"L1": "#2c7bb6", "L2": "#d7191c"}
        all_stages = [s for s in fn.STAGE_ORDER if any(
            s in df["Stage"].values for df in valid_sensors.values()
        )]
        x = np.arange(len(all_stages))
        n_sensors = len(valid_sensors)
        bar_width = 0.35

        for ax, metric, metric_label in zip(axes, bar_metrics, bar_labels):
            for idx, (lbl, df) in enumerate(valid_sensors.items()):
                if metric not in df.columns:
                    continue
                stage_means = (
                    df.groupby("Stage", observed=True)[metric]
                    .mean()
                    .reindex(all_stages)
                )
                offset = (idx - (n_sensors - 1) / 2.0) * bar_width
                ax.bar(
                    x + offset, stage_means.values,
                    bar_width, label=lbl,
                    color=sensor_colours.get(lbl, "grey"),
                    alpha=0.8, edgecolor="white",
                )
            ax.set_xticks(x)
            ax.set_xticklabels(all_stages, rotation=20, ha="right", fontsize=9)
            ax.set_ylabel(metric_label, fontsize=10)
            ax.set_title(metric_label, fontsize=10)
            ax.legend(fontsize=8)
            ax.grid(axis="y", alpha=0.3)

        fig.suptitle(f"Patient {patient_id} — HRV by stage (L1 vs L2)", fontsize=12)
        fig.tight_layout(rect=(0, 0, 1, 0.94))
        save_path = patient_out / "ecg_stage_summary.png"
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()
        logger.info("Saved: %s", save_path)

## 4. Cross-patient summary

In [ ]:
summary_rows = []

for patient_id, data in results.items():
    for label in ("L1", "L2"):
        df = data.get(label)
        if df is None or df.empty:
            continue
        row: dict = {"patient_id": patient_id, "sensor": label}
        for metric in ["hr_bpm", "breathing_rate_bpm", "rmssd_ms",
                        "sdnn_ms", "lf_ms2", "hf_ms2", "lf_hf_ratio"]:
            if metric in df.columns:
                row[metric] = round(float(df[metric].mean(skipna=True)), 2)
        row["n_windows"] = len(df)
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

if not summary_df.empty:
    rename_map = {k: v for k, v in METRIC_LABELS.items() if k in summary_df.columns}
    display_summary = summary_df.rename(columns=rename_map)
    display(display_summary)
else:
    print("No data to summarise.")

## 5. Export

In [ ]:
# Per-patient time-series CSVs.
for patient_id, data in results.items():
    for label in ("L1", "L2"):
        df = data.get(label)
        if df is None or df.empty:
            continue
        csv_path = ecg_out / patient_id / f"ecg_timeseries_{label}.csv"
        df.to_csv(csv_path, index=False)
        logger.info("Saved: %s", csv_path)
        print(f"Saved: {csv_path}")

# Cross-patient summary CSV.
if not summary_df.empty:
    out_csv = ecg_out / "ecg_summary.csv"
    summary_df.to_csv(out_csv, index=False)
    logger.info("Saved summary: %s", out_csv)
    print(f"Saved summary: {out_csv}")
else:
    print("No summary to export.")